# Lazy Predict

LazyClassifier is fit on the data to get an idea of which models are most worth investigating.

Before fitting, the target variable is changed to numerical type and the categorical features are one hot encoded.

The models to investigate further are:
- Nearest Centroid
- XGBoost
- Bagging Classifier
- Gaussian NB

In [2]:
# change directory to the repository
import os
os.chdir(r"C:\Users\mattc\OneDrive\Documents\Data\Apziva\uyBvjh22fVpcb3UX")
os.getcwd()

'C:\\Users\\mattc\\OneDrive\\Documents\\Data\\Apziva\\uyBvjh22fVpcb3UX'

In [3]:
# import all the necessary libraries
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display, Markdown
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from lazypredict.Supervised import LazyClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import recall_score

In [4]:
# set random state for the notebook
random_state = 42
np.random.seed(random_state)

In [6]:
# import the data
data = pd.read_csv("data/processed/processed_data.csv")

# split data into x and y
x = data.select_dtypes(include='object')
y = data['target']

# split data into train and test sets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=random_state, stratify=y)

# one hot encoding categorical columns
cat_features = x.select_dtypes(include='object').columns

encoder = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
], remainder='passthrough')

x_train_processed = encoder.fit_transform(x_train).toarray()
x_test_processed = encoder.transform(x_test).toarray()

# create custom scorer for class 1 recall
def custom_scorer(y_true, y_pred):
    return recall_score(y_true, y_pred, pos_label=1)

# fitting lazy classifier to encoded data and viewing model performances
lzclf = LazyClassifier(verbose=1, ignore_warnings=True, custom_metric=custom_scorer)
models, predictions = lzclf.fit(x_train_processed, x_test_processed, y_train, y_test)
models

  0%|          | 0/31 [00:00<?, ?it/s]

,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Precision,Recall,custom_scorer,Time Taken
Model,,,,,,,,
NearestCentroid,0.884500,0.803190,0.902889,0.901531,0.930160,0.884500,0.708117,0.267982
LinearDiscriminantAnalysis,0.923125,0.745187,0.912332,0.925374,0.927972,0.923125,0.537133,0.721754
DecisionTreeClassifier,0.907875,0.674069,0.677620,0.909390,0.910992,0.907875,0.400691,0.437038
GaussianNB,0.416250,0.669429,0.882363,0.517397,0.928745,0.416250,0.965458,0.182126
XGBClassifier,0.937000,0.669067,0.932646,0.929183,0.926714,0.937000,0.355786,0.656514
ExtraTreeClassifier,0.909750,0.663933,0.670477,0.909750,0.909750,0.909750,0.376511,0.236625
BernoulliNB,0.926625,0.663474,0.900259,0.921175,0.917476,0.926625,0.355786,0.239436
BaggingClassifier,0.928875,0.659910,0.869063,0.922386,0.918542,0.928875,0.345423,1.988850
CalibratedClassifierCV,0.936000,0.655789,0.916039,0.927021,0.924616,0.936000,0.328152,1.856391
